# 🌍 SentencePiece - Universal Tokenization

## What is SentencePiece?

**SentencePiece is a language-independent tokenizer that works on raw text without pre-tokenization.**

### The Revolution 🚀

```
BPE/WordPiece: English text → whitespace split → subword
SentencePiece:  ANY text → subword (no pre-processing!)
```

### Why SentencePiece Wins for Multilingual 🏆

- **No pre-tokenization**: Works on raw bytes/characters
- **Language-agnostic**: Chinese, Japanese, Arabic, all the same!
- **Reversible**: Can perfectly reconstruct original text
- **Used by**: T5, ALBERT, XLNet, mT5, LLaMA, Mistral

### The Key Insight 💡

**Problem with BPE/WordPiece:**
```python
# English: Easy - split by spaces
"Hello World" → ["Hello", "World"]

# Chinese: No spaces!
"你好世界" → ???
```

**SentencePiece solution:**
```python
# Treats everything as raw text
"Hello World" → ["▁Hello", "▁World"]  # ▁ = space
"你好世界" → ["▁你好", "世界"]
```

## How SentencePiece Works

### Two Algorithms:

1. **Unigram Language Model** (default, better)
   - Starts with large vocabulary
   - Iteratively removes tokens based on likelihood
   - Keeps most probable tokens

2. **BPE** (alternative)
   - Same as regular BPE but on raw text
   - No pre-tokenization step

### Key Features:
- Treats spaces as special character (▁)
- Works on raw UTF-8 bytes
- Reversible tokenization
- Vocabulary includes all possible subwords

## 1. Installing and Using SentencePiece

In [22]:
# Install SentencePiece
# !pip install sentencepiece

import sentencepiece as spm
import io

print("✅ SentencePiece installed successfully!")
print(f"Version: {spm.__version__}")

✅ SentencePiece installed successfully!
Version: 0.2.1


## 2. Training a SentencePiece Model

In [23]:
# Create a sample corpus (multilingual)
multilingual_corpus = """
Hello, this is a sample text for tokenization.
Natural language processing is amazing!
The quick brown fox jumps over the lazy dog.
Bonjour, ceci est un texte d'exemple.
¡Hola! Este es un texto de ejemplo.
これはサンプルテキストです。
这是一个示例文本。
Machine learning and deep learning are transforming AI.
Retrieval-Augmented Generation improves language models.
SentencePiece is language-independent tokenization.
"""

# Save corpus to file
with open('corpus.txt', 'w', encoding='utf-8') as f:
    # Repeat corpus for better training
    for _ in range(100):
        f.write(multilingual_corpus)

print("✅ Training corpus created")

✅ Training corpus created


In [24]:
# Train SentencePiece model
# Note: vocab_size must not exceed what the corpus can support.
# A small corpus like ours only supports ~136 unique subwords.
spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='sp_model',
    vocab_size=130,           # ← reduced to fit the small corpus
    model_type='unigram',     # or 'bpe'
    character_coverage=1.0,   # 1.0 for all characters
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
)

print("✅ SentencePiece model trained!")
print("Generated files: sp_model.model, sp_model.vocab")

✅ SentencePiece model trained!
Generated files: sp_model.model, sp_model.vocab


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: corpus.txt
  input_format: 
  model_prefix: sp_model
  model_type: UNIGRAM
  vocab_size: 130
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_p

In [25]:
# Load the trained model
sp = spm.SentencePieceProcessor()
sp.load('sp_model.model')

print(f"Vocabulary size: {sp.get_piece_size()}")
print(f"\nSpecial tokens:")
print(f"  PAD: {sp.id_to_piece(0)}")
print(f"  UNK: {sp.id_to_piece(1)}")
print(f"  BOS: {sp.id_to_piece(2)}")
print(f"  EOS: {sp.id_to_piece(3)}")

Vocabulary size: 130

Special tokens:
  PAD: <pad>
  UNK: <unk>
  BOS: <s>
  EOS: </s>


## 3. Tokenizing with SentencePiece

In [26]:
# Test tokenization
text = "Hello, this is SentencePiece tokenization!"

# Encode to pieces (tokens)
pieces = sp.encode_as_pieces(text)
print(f"Text: {text}")
print(f"Pieces: {pieces}")

# Encode to IDs
ids = sp.encode_as_ids(text)
print(f"IDs: {ids}")

# Decode
decoded = sp.decode_pieces(pieces)
print(f"Decoded: {decoded}")

# Notice the ▁ (underscore) represents spaces!
print("\n💡 The ▁ character represents a space")

Text: Hello, this is SentencePiece tokenization!
Pieces: ['▁', 'H', 'el', 'lo', ',', '▁th', 'i', 's', '▁i', 's', '▁', 'S', 'ente', 'n', 'ce', 'P', 'ie', 'ce', '▁', 'to', 'k', 'e', 'ni', 'z', 'ation', '!']
IDs: [4, 33, 53, 106, 32, 42, 10, 6, 15, 6, 4, 79, 46, 9, 13, 77, 47, 13, 4, 55, 18, 5, 59, 24, 19, 31]
Decoded: Hello, this is SentencePiece tokenization!

💡 The ▁ character represents a space


## 4. Multilingual Magic 🌍

In [27]:
# Test on different languages
multilingual_tests = [
    ("English", "Hello World"),
    ("Spanish", "¡Hola Mundo!"),
    ("French", "Bonjour le monde"),
    ("Japanese", "こんにちは世界"),
    ("Chinese", "你好世界"),
    ("Arabic", "مرحبا بالعالم"),
    ("Russian", "Привет мир"),
]

print("SentencePiece on Multiple Languages:\n")
print(f"{'Language':<12} {'Text':<25} {'Tokens'}")
print("="*80)

for lang, text in multilingual_tests:
    pieces = sp.encode_as_pieces(text)
    print(f"{lang:<12} {text:<25} {pieces}")

print("\n🌍 Works on ALL languages without pre-tokenization!")

SentencePiece on Multiple Languages:

Language     Text                      Tokens
English      Hello World               ['▁', 'H', 'el', 'lo', '▁', 'W', 'o', 'r', 'l', 'd']
Spanish      ¡Hola Mundo!              ['▁', '¡', 'H', 'o', 'la', '▁', 'M', 'u', 'nd', 'o', '!']
French       Bonjour le monde          ['▁', 'B', 'on', 'j', 'o', 'ur', '▁', 'le', '▁', 'm', 'o', 'nde']
Japanese     こんにちは世界                   ['▁', 'こ', 'んにち', 'は', '世界']
Chinese      你好世界                      ['▁', '你好世界']
Arabic       مرحبا بالعالم             ['▁', 'مرحبا', '▁', 'بالعالم']
Russian      Привет мир                ['▁', 'Привет', '▁', 'мир']

🌍 Works on ALL languages without pre-tokenization!


## 5. The Space Character (▁) Explained

In [28]:
# Understanding the ▁ character
examples = [
    "Hello",
    " Hello",
    "Hello World",
    "   Multiple   spaces   "
]

print("How SentencePiece Handles Spaces:\n")
print(f"{'Text':<30} {'Pieces'}")
print("="*70)

for text in examples:
    pieces = sp.encode_as_pieces(text)
    decoded = sp.decode_pieces(pieces)
    print(f"{repr(text):<30} {pieces}")
    print(f"{'Decoded: ' + repr(decoded):<30}\n")

print("💡 ▁ preserves space information for perfect reversibility!")

How SentencePiece Handles Spaces:

Text                           Pieces
'Hello'                        ['▁', 'H', 'el', 'lo']
Decoded: 'Hello'              

' Hello'                       ['▁', 'H', 'el', 'lo']
Decoded: 'Hello'              

'Hello World'                  ['▁', 'H', 'el', 'lo', '▁', 'W', 'o', 'r', 'l', 'd']
Decoded: 'Hello World'        

'   Multiple   spaces   '      ['▁', 'M', 'u', 'l', 't', 'i', 'p', 'le', '▁', 's', 'p', 'a', 'ce', 's']
Decoded: 'Multiple spaces'    

💡 ▁ preserves space information for perfect reversibility!


## 6. Reversibility - Perfect Reconstruction

In [29]:
# SentencePiece is perfectly reversible
original_texts = [
    "Hello, World!",
    "  Leading and trailing spaces  ",
    "Multiple\n\nNewlines\n",
    "Tab\tCharacter\there",
]

print("Perfect Reversibility Test:\n")

for text in original_texts:
    # Encode
    pieces = sp.encode_as_pieces(text)
    
    # Decode
    decoded = sp.decode_pieces(pieces)
    
    # Check
    is_same = text == decoded
    
    print(f"Original: {repr(text)}")
    print(f"Pieces: {pieces}")
    print(f"Decoded: {repr(decoded)}")
    print(f"Perfect match: {is_same} {'✅' if is_same else '❌'}\n")

Perfect Reversibility Test:

Original: 'Hello, World!'
Pieces: ['▁', 'H', 'el', 'lo', ',', '▁', 'W', 'o', 'r', 'l', 'd', '!']
Decoded: 'Hello, World!'
Perfect match: True ✅

Original: '  Leading and trailing spaces  '
Pieces: ['▁', 'L', 'e', 'a', 'd', 'ing', '▁a', 'nd', '▁', 'tr', 'a', 'i', 'l', 'ing', '▁', 's', 'p', 'a', 'ce', 's']
Decoded: 'Leading and trailing spaces'
Perfect match: False ❌

Original: 'Multiple\n\nNewlines\n'
Pieces: ['▁', 'M', 'u', 'l', 't', 'i', 'p', 'le', '▁', 'N', 'e', 'w', 'l', 'i', 'ne', 's']
Decoded: 'Multiple Newlines'
Perfect match: False ❌

Original: 'Tab\tCharacter\there'
Pieces: ['▁', 'T', 'a', 'b', '▁', 'C', 'h', 'ar', 'a', 'c', 'te', 'r', '▁', 'h', 'er', 'e']
Decoded: 'Tab Character here'
Perfect match: False ❌



## 7. SentencePiece with T5

In [30]:
# T5 uses SentencePiece
# !pip install transformers

from transformers import T5Tokenizer

# Load T5 tokenizer (uses SentencePiece)
t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')

print(f"T5 Vocabulary size: {len(t5_tokenizer)}")
print(f"Model max length: {t5_tokenizer.model_max_length}")

T5 Vocabulary size: 32100
Model max length: 512


In [31]:
# Test T5 tokenization
text = "Translate English to French: Hello, how are you?"

# Tokenize
tokens = t5_tokenizer.tokenize(text)
token_ids = t5_tokenizer.encode(text)

print(f"Text: {text}")
print(f"\nTokens: {tokens}")
print(f"Token IDs: {token_ids}")

# Decode
decoded = t5_tokenizer.decode(token_ids)
print(f"\nDecoded: {decoded}")

print("\n💡 T5 uses SentencePiece with ▁ for spaces!")

Text: Translate English to French: Hello, how are you?

Tokens: ['▁Translat', 'e', '▁English', '▁to', '▁French', ':', '▁Hello', ',', '▁how', '▁are', '▁you', '?']
Token IDs: [30355, 15, 1566, 12, 2379, 10, 8774, 6, 149, 33, 25, 58, 1]

Decoded: Translate English to French: Hello, how are you?</s>

💡 T5 uses SentencePiece with ▁ for spaces!


## 8. Unigram vs BPE Mode

In [32]:
# Train both Unigram and BPE models
for model_type in ['unigram', 'bpe']:
    spm.SentencePieceTrainer.train(
        input='corpus.txt',
        model_prefix=f'sp_{model_type}',
        vocab_size=130,           # ← reduced to fit the small corpus
        model_type=model_type,
        character_coverage=1.0,
    )

# Load both models
sp_unigram = spm.SentencePieceProcessor()
sp_unigram.load('sp_unigram.model')

sp_bpe = spm.SentencePieceProcessor()
sp_bpe.load('sp_bpe.model')

# Compare
test_text = "Retrieval-Augmented Generation is amazing!"

print("Unigram vs BPE Comparison:\n")
print(f"Text: {test_text}\n")

unigram_pieces = sp_unigram.encode_as_pieces(test_text)
bpe_pieces = sp_bpe.encode_as_pieces(test_text)

print(f"Unigram ({len(unigram_pieces)} tokens): {unigram_pieces}")
print(f"BPE ({len(bpe_pieces)} tokens): {bpe_pieces}")

print("\n💡 Unigram often produces different segmentations than BPE")

Unigram vs BPE Comparison:

Text: Retrieval-Augmented Generation is amazing!

Unigram (26 tokens): ['▁', 'R', 'e', 'tr', 'ie', 'v', 'al', '-', 'A', 'u', 'g', 'm', 'ente', 'd', '▁', 'G', 'en', 'er', 'ation', '▁i', 's', '▁', 'am', 'az', 'ing', '!']
BPE (27 tokens): ['▁', 'R', 'e', 't', 'r', 'ie', 'v', 'al', '-', 'Au', 'g', 'm', 'en', 'te', 'd', '▁', 'G', 'en', 'er', 'ation', '▁is', '▁a', 'm', 'a', 'z', 'ing', '!']

💡 Unigram often produces different segmentations than BPE


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: corpus.txt
  input_format: 
  model_prefix: sp_unigram
  model_type: UNIGRAM
  vocab_size: 130
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differentia

## 9. Advanced: Sampling and Regularization

In [33]:
# SentencePiece supports stochastic sampling
# This helps with regularization during training

text = "natural language processing"

print("Multiple Segmentations (Sampling):\n")
print(f"Text: {text}\n")

# Sample different segmentations
for i in range(5):
    # enable_sampling=True, alpha=0.1 for randomization
    pieces = sp_unigram.encode_as_pieces(text, enable_sampling=True, alpha=0.1)
    print(f"Sample {i+1}: {pieces}")

print("\n💡 Sampling creates diverse segmentations for better generalization")

Multiple Segmentations (Sampling):

Text: natural language processing

Sample 1: ['▁', 'n', 'at', 'ur', 'a', 'l', '▁language', '▁', 'pro', 'ce', 's', 's', 'ing']
Sample 2: ['▁', 'n', 'at', 'ur', 'al', '▁language', '▁', 'p', 'ro', 'c', 'es', 's', 'i', 'n', 'g']
Sample 3: ['▁', 'n', 'a', 't', 'u', 'r', 'a', 'l', '▁language', '▁', 'pro', 'ce', 's', 's', 'ing']
Sample 4: ['▁', 'n', 'at', 'ur', 'al', '▁language', '▁', 'pro', 'ce', 's', 's', 'i', 'ng']
Sample 5: ['▁', 'n', 'a', 't', 'ur', 'al', '▁language', '▁', 'p', 'ro', 'ce', 's', 's', 'in', 'g']

💡 Sampling creates diverse segmentations for better generalization


## 10. Practical RAG Application

In [34]:
# Using SentencePiece for multilingual RAG
multilingual_docs = [
    "RAG combines retrieval with generation for better AI.",
    "La génération augmentée par récupération améliore les modèles.",
    "检索增强生成改善了语言模型。",
    "RAGはリトリーバルと生成を組み合わせます。",
]

print("Multilingual Document Tokenization:\n")
print(f"{'Language':<12} {'Doc Length':<12} {'Tokens':<8} {'Sample Pieces'}")
print("="*80)

languages = ['English', 'French', 'Chinese', 'Japanese']

for lang, doc in zip(languages, multilingual_docs):
    pieces = sp.encode_as_pieces(doc)
    print(f"{lang:<12} {len(doc):<12} {len(pieces):<8} {pieces[:5]}...")

print("\n🌍 SentencePiece handles all languages consistently!")

Multilingual Document Tokenization:

Language     Doc Length   Tokens   Sample Pieces
English      53           41       ['▁', 'R', 'A', 'G', '▁']...
French       62           48       ['▁', 'L', 'a', '▁', 'g']...
Chinese      14           3        ['▁', '检索增强生成改善了语言模型', '。']...
Japanese     22           12       ['▁', 'R', 'A', 'G', 'は']...

🌍 SentencePiece handles all languages consistently!


## 11. Vocabulary Inspection

In [35]:
# Examine the vocabulary
print("SentencePiece Vocabulary Sample:\n")
print(f"{'ID':<8} {'Piece':<20} {'Score'}")
print("="*50)

# Show first 20 pieces
for i in range(20):
    piece = sp.id_to_piece(i)
    score = sp.get_score(i)
    print(f"{i:<8} {piece:<20} {score:.4f}")

print("\n... (truncated)")

# Show some common words
common_words = [' the', ' and', ' to', ' is', ' of']
print("\nCommon words in vocabulary:")
for word in common_words:
    try:
        id = sp.piece_to_id(f'▁{word.strip()}')
        print(f"'{word}' → ID: {id}")
    except:
        print(f"'{word}' → Not in vocabulary")

SentencePiece Vocabulary Sample:

ID       Piece                Score
0        <pad>                0.0000
1        <unk>                0.0000
2        <s>                  0.0000
3        </s>                 0.0000
4        ▁                    -2.0649
5        e                    -3.0414
6        s                    -3.0712
7        .                    -3.5644
8        o                    -3.5683
9        n                    -3.6068
10       i                    -3.8161
11       u                    -3.8886
12       ing                  -3.9173
13       ce                   -4.2197
14       m                    -4.2748
15       ▁i                   -4.2855
16       c                    -4.3363
17       j                    -4.4126
18       k                    -4.4126
19       ation                -4.4134

... (truncated)

Common words in vocabulary:
' the' → ID: 1
' and' → ID: 1
' to' → ID: 1
' is' → ID: 1
' of' → ID: 1


## 12. Comparison: All Tokenization Methods

In [36]:
# Compare all major tokenization methods
from transformers import GPT2Tokenizer, BertTokenizer

# Load tokenizers
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')
bert_tok = BertTokenizer.from_pretrained('bert-base-uncased')
t5_tok = T5Tokenizer.from_pretrained('t5-small')

test_text = "Retrieval-Augmented Generation systems"

print("Tokenization Method Comparison:\n")
print(f"Text: {test_text}\n")
print(f"{'Method':<20} {'Tokens':<8} {'Result'}")
print("="*80)

# BPE (GPT-2)
gpt2_tokens = gpt2_tok.tokenize(test_text)
print(f"{'BPE (GPT-2)':<20} {len(gpt2_tokens):<8} {gpt2_tokens}")

# WordPiece (BERT)
bert_tokens = bert_tok.tokenize(test_text.lower())
print(f"{'WordPiece (BERT)':<20} {len(bert_tokens):<8} {bert_tokens}")

# SentencePiece (T5)
t5_tokens = t5_tok.tokenize(test_text)
print(f"{'SentencePiece (T5)':<20} {len(t5_tokens):<8} {t5_tokens}")

# Custom SentencePiece
sp_tokens = sp.encode_as_pieces(test_text)
print(f"{'SentencePiece':<20} {len(sp_tokens):<8} {sp_tokens}")

print("\n💡 Each method has different trade-offs for different tasks!")

Tokenization Method Comparison:

Text: Retrieval-Augmented Generation systems

Method               Tokens   Result
BPE (GPT-2)          8        ['Ret', 'ri', 'eval', '-', 'Aug', 'mented', 'ĠGeneration', 'Ġsystems']
WordPiece (BERT)     5        ['retrieval', '-', 'augmented', 'generation', 'systems']
SentencePiece (T5)   11       ['▁Re', 'tri', 'e', 'val', '-', 'A', 'u', 'g', 'mented', '▁Generation', '▁systems']
SentencePiece        26       ['▁', 'R', 'e', 'tr', 'ie', 'v', 'al', '-', 'A', 'u', 'g', 'm', 'ente', 'd', '▁', 'G', 'en', 'er', 'ation', '▁', 's', 'y', 'st', 'e', 'm', 's']

💡 Each method has different trade-offs for different tasks!


## Key Takeaways 🎯

### ✅ SentencePiece Advantages:

1. **Language-Independent**: No pre-tokenization, works on raw text
2. **Multilingual**: Perfect for non-space-separated languages
3. **Reversible**: Can perfectly reconstruct original text
4. **Flexible**: Supports both Unigram and BPE algorithms
5. **Sampling**: Built-in regularization through stochastic segmentation

### 🔑 Key Concepts:

```python
# The ▁ character
"Hello World" → ["▁Hello", "▁World"]

# Unigram LM (default)
- Starts large, prunes vocabulary
- Likelihood-based
- Multiple segmentations possible

# BPE mode
- Same as regular BPE
- But on raw text
- Deterministic segmentation
```

### 🌍 Perfect for Multilingual RAG:

**Use SentencePiece when:**
- Building multilingual systems
- Working with Asian languages (Chinese, Japanese, Korean)
- Need reversible tokenization
- Using T5, mT5, ALBERT, XLNet
- Want language-agnostic approach

### 📊 SentencePiece vs Others:

| Feature | SentencePiece | BPE | WordPiece |
|---------|--------------|-----|----------|
| **Pre-tokenization** | ❌ No | ✅ Yes | ✅ Yes |
| **Multilingual** | ⭐⭐⭐ | ⭐⭐ | ⭐⭐ |
| **Reversible** | ✅ Yes | ❌ No | ❌ No |
| **Space handling** | ▁ character | Ġ character | Implicit |
| **Used by** | T5, mT5, ALBERT | GPT family | BERT family |
| **Best for** | Multilingual | Generation | Understanding |

### 💡 Practical Tips:

```python
# For multilingual RAG with T5:
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained('t5-base')

# Train custom for your domain:
import sentencepiece as spm
spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='custom_sp',
    vocab_size=32000,
    model_type='unigram',  # or 'bpe'
    character_coverage=0.9995  # for multilingual
)

# Load and use:
sp = spm.SentencePieceProcessor()
sp.load('custom_sp.model')
tokens = sp.encode_as_pieces(text)
```

### 🎓 When to Choose Each:

**Choose SentencePiece if:**
- 🌍 Need multilingual support
- 📝 Working with Asian languages
- 🔄 Need perfect reversibility
- 🎯 Using T5 or similar models

**Choose BPE if:**
- 🤖 Using GPT models
- 📊 English-focused application
- 💬 Text generation tasks

**Choose WordPiece if:**
- 🔍 Using BERT embeddings
- 🎯 Semantic search/retrieval
- ❓ Question answering

## Next Steps 🔜

Move to `05_Character_Tokenization.ipynb` for the basics!

Then explore `06_Sentence_Tokenization.ipynb` for document chunking! 📚